In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import warnings
warnings.filterwarnings("ignore")

In [13]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [14]:
import subprocess
import time

# Start the Ollama server as a background process
subprocess.Popen(['ollama', 'serve'])

# Wait a few seconds for the server to initialize
time.sleep(5)
print("Ollama server is running!")

Ollama server is running!


In [15]:
!ollama pull llama3.2

In [7]:
!ollama run llama3.2 "Hello, how are you?"

I'm just a language model, so I don't have feelings or emotions like humans
humans do. However, I'm functioning properly and ready to help with any que
questions or tasks you may have! How can I assist you today?



In [16]:
import glob
import os

# Method 1: If all classes are in one parent folder
base_path = "/content/drive/MyDrive/FACMIC/data_augmented/BrainTumor/"

all_image_paths = []
for client in ['client_0','client_1','client_2','client_3']:
  client_path=os.path.join(base_path,client)
  for class_folder in ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']:
      folder_path = os.path.join(client_path, class_folder)

      # Get all jpg images
      images = glob.glob(os.path.join(folder_path, "*.jpg"))
      all_image_paths.extend(images)

      # Also get png if you have them
      images_png = glob.glob(os.path.join(folder_path, "*.png"))
      all_image_paths.extend(images_png)

print(f"Found {len(all_image_paths)} images")
print(f"First few: {all_image_paths[:3]}")

Found 9335 images
First few: ['/content/drive/MyDrive/FACMIC/data_augmented/BrainTumor/client_0/glioma_tumor/gg (522).jpg', '/content/drive/MyDrive/FACMIC/data_augmented/BrainTumor/client_0/glioma_tumor/gg (457).jpg', '/content/drive/MyDrive/FACMIC/data_augmented/BrainTumor/client_0/glioma_tumor/gg (345).jpg']


In [17]:
!pip install ollama


In [18]:
import ollama
import json
import os
from PIL import Image
import base64
from io import BytesIO

class LLMPromptGenerator:
    """Generate image-specific prompts using local LLM"""

    def __init__(self, model_name="llama3.2-vision"):
        self.model = model_name
        self.client = ollama.Client()

    def image_to_base64(self, image_path):
        """Convert image to base64 for LLM"""
        img = Image.open(image_path)
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        return base64.b64encode(buffered.getvalue()).decode()

    def generate_prompt_for_image(self, image_path, class_name):
        """
        Generate a medical description for a single image
        """

        # Create instruction for the LLM
        instruction = f"""You are a radiologist. Look at this brain MRI image which shows {class_name.replace('_', ' ')}.
Generate a single concise medical description (one sentence, max 15 words) that describes the key radiological findings you observe.
Focus on: location, borders, enhancement pattern, surrounding tissue effects.
Format: "[your description]"
Description:"""

        try:
            # For vision models
            if "vision" in self.model:
                response = self.client.generate(
                    model=self.model,
                    prompt=instruction,
                    images=[image_path]
                )
            else:
                # For text-only models (no image, use class name)
                response = self.client.generate(
                    model=self.model,
                    prompt=f"""Generate a realistic medical description for a brain MRI showing {class_name.replace('_', ' ')}.

Examples:
- "brain MRI showing infiltrative glioma with irregular borders"
- "brain MRI showing extra-axial meningioma with dural attachment"
- "brain MRI showing sellar pituitary adenoma"

Generate one similar description (max 15 words):"""
                )

            # Extract and clean the response
            description = response['response'].strip('"')

            # Ensure it starts with "brain MRI showing"

            # if not description.lower().startswith("brain mri"):
            #     description = f"brain MRI showing {description}"

            return description

        except Exception as e:
            print(f"Error generating prompt: {e}")
            return f"brain MRI showing {class_name.replace('_', ' ')}"

    def generate_all_prompts(self, image_paths, save_path='/content/drive/MyDrive/FACMIC/llm_prompts_augmented_dataset.json'):
        """Generate prompts for all images"""
        prompts = {}

        print(f"Generating prompts for {len(image_paths)} images...")

        for i, img_path in enumerate(image_paths):
            class_name = os.path.basename(os.path.dirname(img_path))

            prompt = self.generate_prompt_for_image(img_path, class_name)
            prompts[img_path] = prompt

            if (i + 1) % 10 == 0:
                print(f"Progress: {i+1}/{len(image_paths)}")

        # Save prompts
        with open(save_path, 'w') as f:
            json.dump(prompts, f, indent=2)

        print(f"Saved {len(prompts)} prompts to {save_path}")
        return prompts

# Usage
generator = LLMPromptGenerator(model_name="llama3.2")

# Generate prompts for all images
# all_image_paths = [...]  # Your image paths
prompts = generator.generate_all_prompts(all_image_paths)

Generating prompts for 9335 images...
Progress: 10/9335
Progress: 20/9335
Progress: 30/9335
Progress: 40/9335
Progress: 50/9335
Progress: 60/9335
Progress: 70/9335
Progress: 80/9335
Progress: 90/9335
Progress: 100/9335
Progress: 110/9335
Progress: 120/9335
Progress: 130/9335
Progress: 140/9335
Progress: 150/9335
Progress: 160/9335
Progress: 170/9335
Progress: 180/9335
Progress: 190/9335
Progress: 200/9335
Progress: 210/9335
Progress: 220/9335
Progress: 230/9335
Progress: 240/9335
Progress: 250/9335
Progress: 260/9335
Progress: 270/9335
Progress: 280/9335
Progress: 290/9335
Progress: 300/9335
Progress: 310/9335
Progress: 320/9335
Progress: 330/9335
Progress: 340/9335
Progress: 350/9335
Progress: 360/9335
Progress: 370/9335
Progress: 380/9335
Progress: 390/9335
Progress: 400/9335
Progress: 410/9335
Progress: 420/9335
Progress: 430/9335
Progress: 440/9335
Progress: 450/9335
Progress: 460/9335
Progress: 470/9335
Progress: 480/9335
Progress: 490/9335
Progress: 500/9335
Progress: 510/9335
Pr